## **Data cleaning**

---

**TODO**

- 🔄️ Handle missing days
    - Add 8 missing days as empty rows, or keep it being removed ???
- 🔄️ Handle 2021 observations 
    - Roughly half the proportion's real reading and the other's NaN
- 🔄️ (regression only) Drop `Observation_ID` if no real data near 2026-08-31 

1. `Library`
2. `Function`:<br>
   (i) Reshape<br>
   (ii) Clean error<br>
   (iii) Drop IDs
3. `Load data`
4. `Pass function`
5. `Visualization`

In [1]:
# Data libraries

import pandas as pd
import numpy as np

> ### Function 1: Reshape

- ✅ Reshape wide to long (one row per `Observation_ID` per day) + merge avg/min/max + location
- ✅ Convert data headers to real dates (day/month/year)
- ✅ Convert text labels (`County`, `Station_Name`, `Observation_ID`) as categories
- ✅ Remove rows with 'tidal-level' observation (irrelevant to 'river' obs.)<br>
  → (default) `drop_tidal = True`
- ✅ Remove data before 2017<br>
  → (default) `start = 2017-04-01`
    - 2014-2016: no error codes, mostly NaN
    - real data appear to start ~2017-04-01<br>
  (i.e., NaN share by month drops a lot from April/2017; worth considering as start-date)

In [2]:
def reshape_long(avg_df, min_df, max_df, loc_df, drop_tidal = True, start = "2017-04-01"):
    
    data = [avg_df, min_df, max_df]  
    label = ["avg", "min", "max"]
    column = ["County", "Station_Name", "Observation_ID", "date"]
    long_df = {}

    
    for df, lab in zip(data, label):

        # (1) drop tidal-level observation
        if drop_tidal:
            df = df[~df["Observation_ID"].str.contains("tidal-level", na = False)]  
        
        # (2) wide to long (one row per Observation_ID per day)
        long_df[lab] = df.melt(id_vars = column[0:3],
                               var_name = "date",
                               value_name = lab)

        # (3) convert date headers to real dates 
        long_df[lab]["date"] = pd.to_datetime(long_df[lab]["date"], format = "%d/%m/%Y")

        # (4) drop days before 'start'
        long_df[lab] = long_df[lab][long_df[lab]["date"] >= start]

    
    # (5) combine 3 long tables (avg,min,max) + location df
    river_long = (long_df["avg"]
                 .merge(long_df["min"], on = column, how = "left")
                 .merge(long_df["max"], on = column, how = "left")
                 .merge(loc_df, on = column[0:3], how = "left"))

    
    # (6) convert text labels as categories
    for col in column[0:3]:
        river_long[col] = river_long[col].astype("category")

    
    return river_long

> ### Function 2: Clean error

- ✅ Handle error codes
    - error code: 99999.999, 999999.999, -999999.999, -99
    - rule: if any of avg/min/max has abs ≧ 9999 (or is a code) → blank all three for that day
    - 78 error days removed
- ✅ Handle outliers
    - rule: if outside per-station IQR fence (q1 - k * IQR ... or ... q3 + k * IQR) → **NaN**
    - `k=20`: very wide, only placeholders (= extremes) are removed
    - outliers removed: avg 113, min 179, max 146

In [3]:
def clean_error(df, k = 20):
    # k = IQR multiplier of choice (set glitch fence)
    
    label = ["avg", "min", "max"]
    df2 = df.copy()

    # handling error codes
    err = (df2[label].isin([99999.999, -999999.999, -99]) | (df2[label].abs() >= 9999)).any(axis = 1)
    df2.attrs["error"] = int(err.sum())

    df2.loc[err, label] = np.nan

    # handling outliers
    for lab in label:
        per_sta = df2.groupby("Observation_ID", observed = True)[lab]
        
        q1 = per_sta.transform(lambda s: s.quantile(0.25))
        q3 = per_sta.transform(lambda s: s.quantile(0.75))
        iqr = q3 - q1
        
        outlier = ((df2[lab] > q3 + k * iqr) | (df2[lab] < q1 - k * iqr))
        df2.attrs[f"outliers_{lab}"] = int(outlier.sum())
        
        df2.loc[outlier, lab] = np.nan
    
    return df2

> ### Function 3: Drop IDs

- ✅ Drop IDs with little observation
    - real (non-NaN) avg days / 365.25 < 3 years → whole `Observation_ID` dropped<br>
      <u>NOTE.</u> (365x3 + 366x1) / 4 = 365.25 yrs 
    - 8 IDs dropped, 372 IDs kept

In [4]:
def drop_ids(df, min_years = 3):
    years = df["avg"].notna().groupby(df["Observation_ID"], observed = True).sum() / 365.25
        # avg(4yrs) = (365*3 + 366*1)/4 = 365.25 days

    keep_ids = years[years >= min_years].index
    keep_df = df[df["Observation_ID"].isin(keep_ids)].copy()

#    keep_df["Observation_ID"] = keep_df["Observation_ID"].cat.remove_unused_categories()

    keep_df.attrs["dropped_ids"] = int((years < min_years).sum())
    keep_df.attrs["kept_ids"] = int((years >= min_years).sum())

    return keep_df

In [ ]:
# Load data

river_avg = pd.read_csv("data/avg_river_levels.csv")
river_min = pd.read_csv("data/min_river_levels.csv")
river_max = pd.read_csv("data/max_river_levels.csv")

river_location = pd.read_csv("data/monitoring_locations.csv")

In [ ]:
# function 1: reshape 
river_long = reshape_long(river_avg, river_min, river_max, river_location)
river_long.shape()

In [ ]:
# function 2: clean error 
river_long2 = clean_error(river_long)

In [ ]:
# function 3: drop ids
river_long3 = drop_ids(river_long2)

In [ ]:
# .attrs ... store 'error', 'outliers_avg/min/max', 'dropped/kept_ids'
river_long3.attrs

> ### Visualization

- Scatter plot:
    - before vs. after error cleaning
- Bar chart:
    - real vs. NaN counts per year (avg/min/max)
- Line plot:
    - NaN share by month (shows drop from April/2017)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize = (14,4))

# before error cleaning 
ax[0].scatter(river_long["date"], river_long["min"], s = 1, alpha = 1)
ax[0].set_title("with error codes")

# after error cleaning 
ax[1].scatter(river_long2["date"], river_long2["min"], s = 1, alpha = 1)
ax[1].set_title("without error codes")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize = (16, 4), sharey = True)

for ax, col in zip(axes, ["avg", "min", "max"]):
    status = np.where(river_long2[col].isna(), "NaN", "real")
    counts = pd.crosstab(river_long2["date"].dt.year, status)
    
    ax.set_title(col)
    counts[["real", "NaN"]].plot(kind = "bar", ax = ax, width = 0.8, color = ["tab:blue", "tab:red"], 
                                    rot = 45, legend = False)
    
fig.legend(["Real readings", "Missing (NaN)"], 
           loc = "lower center", ncol = 2, bbox_to_anchor = (0.5, 1.0))

plt.tight_layout()
plt.show()

In [ ]:
# proportion of NaNs over months
river_jan2017 = reshape_long(river_avg, river_min, river_max, river_location, 
                                  start = "2017-01-01")

river_jan2017_month = river_jan2017["avg"].isna().groupby(river_jan2017["date"].dt.to_period("M")).mean()

# plot 
fig, ax = plt.subplots(figsize = (12, 4))

river_jan2017_month.plot(ax = ax)

ax.axvline(pd.Period("2017-04", freq = "M"), 
           color = "orange", linestyle = "--", label = "Apr 2017 (possible start)")
ax.set_ylabel("Proportion of NaN")
ax.set_title("Monthly NaN proportion from Jan 2017")
ax.legend()

plt.show()